In [43]:
import os
import pandas as pd
from pathlib import Path
import gc
import torch

# Run garbage collection
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Support Kaggle and local environments
kaggle_test_path = '/kaggle/input/competitions/name-that-llm/test.csv'
local_test_path = r'C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\data\train.csv'
# local_test_path = r'C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\data\test.csv'

if os.path.exists(kaggle_test_path):
    data_path = kaggle_test_path
elif os.path.exists(local_test_path):
    data_path = local_test_path
elif os.path.exists('data/test.csv'):
    data_path = 'data/test.csv'
elif os.path.exists(local_test_path):
    data_path = local_test_path
else:
    data_path = 'test.csv'

print(f'Loading test data from: {data_path}')
test_df = pd.read_csv(data_path)
print(f'Test samples: {len(test_df)}')
test_df.head()

Loading test data from: C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\data\train.csv
Test samples: 2699


,id,prompt_id,text,model
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",qwen3.8-max
1,0041ac86ee93,p073,Data science communities matter because they a...,claude-sonnet-5
2,005a19083eae,p120,"In the final twilight of time, the last embers...",gemini-3.6-flash
3,009319596f10,p073,Data science communities matter because they t...,gpt-5.6-terra
4,00d3e8c92596,p097,"Deep inside each ear, three tiny loops of bone...",glm-5.3-flash


In [44]:
import os
import torch
import joblib
import pickle
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ========================================================
# Dynamic Model Directory Path Setup
# Someone can paste any directory path here to load models
# ========================================================
SAVE_DIR = r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\ensemble_pipeline_1"

# Dynamic fallback check if path is changed or running in Kaggle
if not os.path.exists(SAVE_DIR):
    candidates = [
        "/kaggle/input/ensemble-pipeline-1",
        "/kaggle/input/ensemble-pipeline",
        r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\ensemble_pipeline",
        "ensemble_pipeline_1",
    ]
    for c in candidates:
        if os.path.exists(c):
            SAVE_DIR = c
            break

print(f"Using pipeline directory: {SAVE_DIR}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Load DeBERTa tokenizer & model
deberta_path = os.path.join(SAVE_DIR, "deberta")
print(f"Loading DeBERTa from: {deberta_path}")
tokenizer = AutoTokenizer.from_pretrained(deberta_path)

model_kwargs = {}
if torch.cuda.is_available():
    model_kwargs["torch_dtype"] = torch.float16

model = AutoModelForSequenceClassification.from_pretrained(deberta_path, **model_kwargs)
model.to(device)
model.eval()

# 2. Load Logistic Regression Classifier & Feature Scaler (encoder.pkl)
LR_DIR = os.path.join(SAVE_DIR, "logistic_regression")
lr_model_path = os.path.join(LR_DIR, "classifier.joblib")
scaler_path = os.path.join(LR_DIR, "encoder.pkl")

print(f"Loading Logistic Regression from: {lr_model_path}")
clf = joblib.load(lr_model_path)

print(f"Loading feature scaler (encoder.pkl) from: {scaler_path}")
with open(scaler_path, "rb") as f:
    encoder = pickle.load(f)

# 3. Load Ensemble Configuration
cfg_path = os.path.join(SAVE_DIR, "ensemble_config.joblib")
if os.path.exists(cfg_path):
    config = joblib.load(cfg_path)
    print("Loaded ensemble config:", config)
else:
    config = {"deberta_weight": 0.5, "lr_weight": 0.5, "max_length": 256}

print("All models, scaler, and configs loaded successfully!")


Using pipeline directory: C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\ensemble_pipeline_1
Using device: cpu
Loading DeBERTa from: C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\ensemble_pipeline_1\deberta


Loading weights: 100%|██████████| 106/106 [00:00<00:00, 5213.74it/s]

Loading Logistic Regression from: C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\ensemble_pipeline_1\logistic_regression\classifier.joblib
Loading feature scaler (encoder.pkl) from: C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\ensemble_pipeline_1\logistic_regression\encoder.pkl
Loaded ensemble config: {'deberta_weight': 0.5, 'lr_weight': 0.5, 'max_length': 256}
All models, scaler, and configs loaded successfully!



c:\Users\ps302\anaconda3\envs\genai\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\ps302\anaconda3\envs\genai\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [45]:
import os
import pickle

# Dynamically load le.pkl from SAVE_DIR
le_path = os.path.join(SAVE_DIR, "le.pkl")

if not os.path.exists(le_path):
    candidates = [
        "/kaggle/input/ensemble-pipeline-1/le.pkl",
        "/kaggle/input/deberta-v3-finetuned/le.pkl",
        r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\deberta_v3_finetuned\le.pkl",
        "le.pkl"
    ]
    for c in candidates:
        if os.path.exists(c):
            le_path = c
            break

print(f"Loading label encoder from: {le_path}")
with open(le_path, "rb") as f:
    le = pickle.load(f)

print(f"Label classes ({len(le.classes_)}):", le.classes_)


Loading label encoder from: C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\ensemble_pipeline_1\le.pkl
Label classes (18): ['claude-fable-5' 'claude-opus-4-6' 'claude-opus-4-8' 'claude-opus-5'
 'claude-sonnet-5' 'gemini-3.1-pro-preview' 'gemini-3.6-flash'
 'gemini-3.7-flash' 'gemini-3.8-flash' 'glm-5.3' 'glm-5.3-flash' 'gpt-5.5'
 'gpt-5.6-luna' 'gpt-5.6-sol' 'gpt-5.6-terra' 'gpt-6-astra' 'kimi-k3'
 'qwen3.8-max']


In [46]:
import os
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

kaggle_model_dir = '/kaggle/input/ensemble-pipeline-1/deberta'
local_model_dir = r'C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\ensemble_pipeline_1\deberta'

if os.path.exists(kaggle_model_dir) and os.path.exists(os.path.join(kaggle_model_dir, 'config.json')):
    LOCAL_DIR = kaggle_model_dir
elif os.path.exists('ensemble_pipeline_1/deberta'):
    LOCAL_DIR = 'ensemble_pipeline_1/deberta'
elif os.path.exists(local_model_dir):
    LOCAL_DIR = local_model_dir
elif os.path.exists(r'C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\ensemble_pipeline\deberta'):
    LOCAL_DIR = r'C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\ensemble_pipeline\deberta'
else:
    LOCAL_DIR = 'ensemble_pipeline/deberta'

print(f'Loading model from: {LOCAL_DIR}')
tokenizer = AutoTokenizer.from_pretrained(LOCAL_DIR)

model_kwargs = {}
if torch.cuda.is_available():
    model_kwargs['torch_dtype'] = torch.float16

model = AutoModelForSequenceClassification.from_pretrained(LOCAL_DIR, **model_kwargs)
model.to(device)
model.eval()
print('Model loaded successfully.')


Using device: cpu
Loading model from: ensemble_pipeline_1/deberta


Loading weights: 100%|██████████| 106/106 [00:00<00:00, 6742.33it/s]

Model loaded successfully.


In [47]:
# Memory-safe batch inference function to prevent OOM errors on large test sets
from torch.utils.data import DataLoader, Dataset

class TextDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return str(self.texts[idx])

dataset = TextDataset(test_df['text'].tolist())


In [48]:
import gc
import torch
import numpy as np
from tqdm.auto import tqdm

BATCH_SIZE = 32
MAX_LENGTH = config.get("max_length", 256)

dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

deberta_probs = []

print(f'Running DeBERTa inference on {len(dataset)} items with batch_size={BATCH_SIZE}...')
with torch.no_grad():
    for batch_texts in tqdm(dataloader, desc='DeBERTa Predicting'):
        encoded = tokenizer(
            list(batch_texts),
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors='pt'
        )
        
        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
        deberta_probs.extend(probs)
        
        del encoded, input_ids, attention_mask, outputs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

deberta_probs = np.array(deberta_probs)
print(f'DeBERTa probability shape: {deberta_probs.shape}')

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Running DeBERTa inference on 2699 items with batch_size=32...


DeBERTa Predicting: 100%|██████████| 85/85 [07:45<00:00,  5.48s/it]


DeBERTa probability shape: (2699, 18)


In [49]:
# predictions_f = le.inverse_transform(all_preds)

# submission = pd.DataFrame({
#     'id': test_df['id'],
#     'model': predictions_f
# })

# # Save output
# submission.to_parquet('submission.parquet', index=False)
# print(f'Saved submission.parquet with {len(submission)} rows.')
# submission.head(10)

In [50]:
# import re
# import numpy as np
# import pandas as pd
# import nltk
# nltk.download("punkt", quiet=True)
# from nltk.tokenize import sent_tokenize, word_tokenize
# from sklearn.preprocessing import StandardScaler

# text_col = "text"

# print(f"Extracting stylometric features on {len(test_df)} test samples...")

# features = pd.DataFrame(index=test_df.index)

# # 1. Character, word, and sentence counts
# features["char_count"] = test_df[text_col].apply(len)

# def get_words_and_sents(text):
#     sents = sent_tokenize(text) if len(text.strip()) > 0 else [""]
#     words = word_tokenize(text) if len(text.strip()) > 0 else [""]
#     return len(words), len(sents), sents, words

# tokenized_info = test_df[text_col].apply(get_words_and_sents)
# features["word_count"] = tokenized_info.apply(lambda x: x[0])
# features["sent_count"] = tokenized_info.apply(lambda x: x[1])

# # Sentence Length Mean & Variance
# def sent_stats(info):
#     _, num_sents, sents, _ = info
#     if num_sents == 0:
#         return 0.0, 0.0
#     lengths = [len(s.split()) for s in sents]
#     return float(np.mean(lengths)), float(np.var(lengths))

# sent_metrics = tokenized_info.apply(sent_stats)
# features["avg_sent_len"] = sent_metrics.apply(lambda x: x[0])
# features["var_sent_len"] = sent_metrics.apply(lambda x: x[1])

# # Avg Word Length
# features["avg_word_len"] = features["char_count"] / (features["word_count"] + 1e-5)

# # 2. Markdown Stylometry
# features["md_asterisk_count"] = test_df[text_col].apply(lambda x: x.count("*"))

# # 3. Punctuation Fingerprint
# features["punct_colon"] = test_df[text_col].apply(lambda x: x.count(":"))
# features["punct_semicolon"] = test_df[text_col].apply(lambda x: x.count(";"))
# features["punct_em_dash"] = test_df[text_col].apply(lambda x: x.count("—") + x.count("--"))
# features["punct_exclamation"] = test_df[text_col].apply(lambda x: x.count("!"))
# features["punct_question"] = test_df[text_col].apply(lambda x: x.count("?"))
# features["punct_comma"] = test_df[text_col].apply(lambda x: x.count(","))
# features["punct_period"] = test_df[text_col].apply(lambda x: x.count("."))
# features["punct_quotes"] = test_df[text_col].apply(lambda x: x.count('"') + x.count("'"))
# features["punct_parens"] = test_df[text_col].apply(lambda x: x.count("(") + x.count(")"))
# features["punct_brackets"] = test_df[text_col].apply(lambda x: x.count("[") + x.count("]"))

# # Punctuation Density (normalized by word count)
# total_puncts = (features["punct_colon"] + features["punct_semicolon"] + 
#                 features["punct_comma"] + features["punct_period"] + features["punct_exclamation"])
# features["punct_density"] = total_puncts / (features["word_count"] + 1e-5)

# # 4. Capitalization & Casing
# features["upper_char_count"] = test_df[text_col].apply(lambda x: sum(1 for c in x if c.isupper()))
# features["upper_char_ratio"] = features["upper_char_count"] / (features["char_count"] + 1e-5)
# features["all_caps_words"] = test_df[text_col].apply(lambda x: len(re.findall(r"\b[A-Z]{2,}\b", x)))
# features["camel_case_count"] = test_df[text_col].apply(lambda x: len(re.findall(r"\b[a-z]+[A-Z][a-zA-Z]*\b", x)))

# # 5. Lexical Diversity (Type-Token Ratio)
# def type_token_ratio(info):
#     num_words, _, _, words = info
#     if num_words == 0:
#         return 0.0
#     unique_words = set(w.lower() for w in words if w.isalnum())
#     return len(unique_words) / (num_words + 1e-5)

# features["lexical_diversity_ttr"] = tokenized_info.apply(type_token_ratio)

# # Digits / Numbers presence
# features["digit_count"] = test_df[text_col].apply(lambda x: sum(1 for c in x if c.isdigit()))
# features["digit_ratio"] = features["digit_count"] / (features["char_count"] + 1e-5)

# print(f"Extracted feature matrix shape: {features.shape}")
# features.head()


In [51]:
import re
import gc
import numpy as np
import pandas as pd
import nltk
from tqdm.auto import tqdm

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import sent_tokenize, word_tokenize

text_col = "text"
texts = test_df[text_col].tolist()
total_samples = len(texts)

# Batch size for memory-safe feature extraction
FEATURE_BATCH_SIZE = 128

print(f"Extracting stylometric features on {total_samples} samples in batches of {FEATURE_BATCH_SIZE}...")

# Pre-compile exact regex patterns
RE_ALL_CAPS = re.compile(r"\b[A-Z]{2,}\b")
RE_CAMEL_CASE = re.compile(r"\b[a-z]+[A-Z][a-zA-Z]*\b")

rows = []

num_batches = int(np.ceil(total_samples / FEATURE_BATCH_SIZE))

for b_idx in tqdm(range(num_batches), desc="Extracting features"):
    start_i = b_idx * FEATURE_BATCH_SIZE
    end_i = min(start_i + FEATURE_BATCH_SIZE, total_samples)
    batch_texts = texts[start_i:end_i]
    
    for text in batch_texts:
        text_str = str(text) if pd.notna(text) else ""
        char_count = len(text_str)
        
        # --- Exact same NLTK tokenization ---
        stripped = text_str.strip()
        sents = sent_tokenize(text_str) if len(stripped) > 0 else [""]
        words = word_tokenize(text_str) if len(stripped) > 0 else [""]
        word_count = len(words)
        sent_count = len(sents)
        
        # --- Sentence Length Mean & Variance ---
        if sent_count == 0:
            avg_sent_len, var_sent_len = 0.0, 0.0
        else:
            lengths = [len(s.split()) for s in sents]
            avg_sent_len = float(np.mean(lengths))
            var_sent_len = float(np.var(lengths))
            
        avg_word_len = char_count / (word_count + 1e-5)
        
        # --- Markdown Stylometry ---
        md_asterisk_count = text_str.count("*")
        
        # --- Punctuation Fingerprint ---
        punct_colon = text_str.count(":")
        punct_semicolon = text_str.count(";")
        punct_em_dash = text_str.count("—") + text_str.count("--")
        punct_exclamation = text_str.count("!")
        punct_question = text_str.count("?")
        punct_comma = text_str.count(",")
        punct_period = text_str.count(".")
        punct_quotes = text_str.count('"') + text_str.count("'")
        punct_parens = text_str.count("(") + text_str.count(")")
        punct_brackets = text_str.count("[") + text_str.count("]")
        
        total_puncts = punct_colon + punct_semicolon + punct_comma + punct_period + punct_exclamation
        punct_density = total_puncts / (word_count + 1e-5)
        
        # --- Capitalization & Casing ---
        upper_char_count = sum(1 for c in text_str if c.isupper())
        upper_char_ratio = upper_char_count / (char_count + 1e-5)
        all_caps_words = len(RE_ALL_CAPS.findall(text_str))
        camel_case_count = len(RE_CAMEL_CASE.findall(text_str))
        
        # --- Lexical Diversity (Type-Token Ratio) ---
        if word_count == 0:
            lexical_diversity_ttr = 0.0
        else:
            unique_words = set(w.lower() for w in words if w.isalnum())
            lexical_diversity_ttr = len(unique_words) / (word_count + 1e-5)
            
        # --- Digits ---
        digit_count = sum(1 for c in text_str if c.isdigit())
        digit_ratio = digit_count / (char_count + 1e-5)
        
        rows.append([
            char_count, word_count, sent_count,
            avg_sent_len, var_sent_len, avg_word_len,
            md_asterisk_count,
            punct_colon, punct_semicolon, punct_em_dash, punct_exclamation, punct_question,
            punct_comma, punct_period, punct_quotes, punct_parens, punct_brackets,
            punct_density,
            upper_char_count, upper_char_ratio, all_caps_words, camel_case_count,
            lexical_diversity_ttr,
            digit_count, digit_ratio
        ])
    
    # Periodic garbage collection every 10 batches to keep memory flat
    if (b_idx + 1) % 10 == 0:
        gc.collect()

feature_cols = [
    "char_count", "word_count", "sent_count",
    "avg_sent_len", "var_sent_len", "avg_word_len",
    "md_asterisk_count",
    "punct_colon", "punct_semicolon", "punct_em_dash", "punct_exclamation", "punct_question",
    "punct_comma", "punct_period", "punct_quotes", "punct_parens", "punct_brackets",
    "punct_density",
    "upper_char_count", "upper_char_ratio", "all_caps_words", "camel_case_count",
    "lexical_diversity_ttr",
    "digit_count", "digit_ratio"
]

features = pd.DataFrame(rows, columns=feature_cols, index=test_df.index)
print(f"Extracted feature matrix shape: {features.shape}")

gc.collect()
features.head()


Extracting stylometric features on 2699 samples in batches of 128...


Extracting features: 100%|██████████| 22/22 [00:06<00:00,  3.54it/s]


Extracted feature matrix shape: (2699, 25)


,char_count,word_count,sent_count,avg_sent_len,var_sent_len,avg_word_len,md_asterisk_count,punct_colon,punct_semicolon,punct_em_dash,...,punct_parens,punct_brackets,punct_density,upper_char_count,upper_char_ratio,all_caps_words,camel_case_count,lexical_diversity_ttr,digit_count,digit_ratio
0,1143,222,8,24.000000,91.000000,5.148648,0,1,2,0,...,0,0,0.126126,10,0.008749,0,0,0.617117,0,0.0
1,518,74,4,16.250000,10.687500,6.999999,0,0,0,0,...,0,0,0.121622,4,0.007722,0,0,0.783784,0,0.0
2,474,89,4,19.250000,17.687500,5.325842,0,0,0,0,...,0,0,0.134831,4,0.008439,0,0,0.707865,0,0.0
3,1724,263,9,24.444444,17.135802,6.555133,0,1,1,0,...,0,0,0.163498,9,0.005220,0,0,0.669201,0,0.0
4,541,114,4,24.250000,8.687500,4.745614,0,1,0,2,...,0,0,0.114035,4,0.007394,0,0,0.684210,0,0.0


In [52]:
# Scale features using the trained encoder.pkl
X_test = encoder.transform(features)

# Compute Logistic Regression probabilities
lr_probs = clf.predict_proba(X_test)
print("Logistic Regression probability shape:", lr_probs.shape)


Logistic Regression probability shape: (2699, 18)


In [53]:
# ==========================================
# Soft Voting Ensemble
# ==========================================
w_deberta = config.get("deberta_weight", 0.5)
w_lr = config.get("lr_weight", 0.5)

print(f"Ensemble weights: DeBERTa={w_deberta:.2f}, Logistic Regression={w_lr:.2f}")

ensemble_probs = (
    w_deberta * deberta_probs +
    w_lr * lr_probs
)

ensemble_preds = np.argmax(ensemble_probs, axis=1)

# If ground truth labels exist in test_df (e.g. validation), evaluate
if "model" in test_df.columns:
    from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
    
    y_true = test_df["model"].values
    if isinstance(y_true[0], str) and hasattr(le, 'transform'):
        y_true = le.transform(y_true)
        
    print("\n========== ENSEMBLE EVALUATION ==========")
    print("Accuracy:", accuracy_score(y_true, ensemble_preds))
    print("Macro Recall:", recall_score(y_true, ensemble_preds, average="macro"))
    print("Confusion Matrix:\n", confusion_matrix(y_true, ensemble_preds))

# Inverse transform predictions to original model labels
predictions_f = le.inverse_transform(ensemble_preds)

# Create submission DataFrame
submission = pd.DataFrame({
    "id": test_df["id"],
    "model": predictions_f
})

# Save output
submission.to_parquet("submission.parquet", index=False)
print(f"\nSaved submission.parquet with {len(submission)} rows.")
submission.head(10)


Ensemble weights: DeBERTa=0.50, Logistic Regression=0.50

========== ENSEMBLE EVALUATION ==========
Accuracy: 0.7050759540570581
Macro Recall: 0.7051006711409395
Confusion Matrix:
 [[115   1  11   1   6   0   0   0   0   1  10   0   1   0   0   2   1   0]
 [  0 135  12   0   1   0   0   0   0   1   1   0   0   0   0   0   0   0]
 [  1   3 135   1   6   0   0   0   0   0   2   0   0   0   0   0   0   2]
 [  2   0   0 136   0   0   0   0   0   5   6   0   0   0   0   1   0   0]
 [  2  14  23   0 108   0   1   0   0   1   1   0   0   0   0   0   0   0]
 [  0   1   0   0   0 143   6   0   0   0   0   0   0   0   0   0   0   0]
 [  0   0   1   0   0  55  92   1   0   0   0   0   0   0   0   0   0   1]
 [  0   0   0   0   0   6  36 105   2   0   0   0   0   0   0   0   0   1]
 [  0   0   1   0   0   3  20  40  86   0   0   0   0   0   0   0   0   0]
 [ 15   3   9   2   1   0   0   0   0  82  33   0   1   1   0   0   1   2]
 [ 38   3   6   2   4   1   0   0   0   9  83   0   0   0   0   3   1

,id,model
0,0018b3765972,qwen3.8-max
1,0041ac86ee93,claude-opus-4-8
2,005a19083eae,gemini-3.6-flash
3,009319596f10,gpt-5.6-terra
4,00d3e8c92596,claude-fable-5
5,00f73c3a3937,gemini-3.1-pro-preview
6,010967f56aaf,gemini-3.8-flash
7,010b4f492abc,claude-opus-4-6
8,013e6e0913db,gemini-3.1-pro-preview
9,0149bc156d4e,gpt-5.6-luna


In [54]:
from sklearn.metrics import balanced_accuracy_score, accuracy_score, recall_score
import numpy as np

# y_true: actual labels
# ensemble_preds: final ensemble predictions

balanced_acc = balanced_accuracy_score(y_true, ensemble_preds)
macro_recall = recall_score(y_true, ensemble_preds, average="macro")
accuracy = accuracy_score(y_true, ensemble_preds)

print(f"Balanced Accuracy: {balanced_acc:.4f}")
print(f"Macro Recall:      {macro_recall:.4f}")
print(f"Accuracy:          {accuracy:.4f}")

# Both metrics should be equal
assert np.isclose(balanced_acc, macro_recall)

Balanced Accuracy: 0.7051
Macro Recall:      0.7051
Accuracy:          0.7051


In [55]:
from sklearn.metrics import balanced_accuracy_score
import numpy as np

# Individual model predictions
deberta_preds = np.argmax(deberta_probs, axis=1)
lr_preds = np.argmax(lr_probs, axis=1)
ensemble_preds = np.argmax(ensemble_probs, axis=1)

# Compare performance
print(f"DeBERTa Balanced Accuracy:   {balanced_accuracy_score(y_true, deberta_preds):.4f}")
print(f"Logistic Regression:         {balanced_accuracy_score(y_true, lr_preds):.4f}")
print(f"Ensemble Balanced Accuracy:  {balanced_accuracy_score(y_true, ensemble_preds):.4f}")

DeBERTa Balanced Accuracy:   0.6955
Logistic Regression:         0.2949
Ensemble Balanced Accuracy:  0.7051
